In [3]:
from pathlib import Path
import numpy as np

# Find the repo root, even if the notebook is in a subfolder.
repo = Path.cwd()
while not (repo / "data/ready_full/fold0").exists():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

for split in ("train", "val"):
    mask = np.load(repo / f"data/ready_full/fold0/mask_{split}.npy")

    # Left padding means each row can only go from 0 to 1, never 1 to 0.
    binary = np.all((mask == 0) | (mask == 1))
    left_padded = np.all(np.diff(mask, axis=1) >= 0, axis=1)
    has_real_steps = np.any(mask == 1, axis=1)

    print(f"{split}: shape={mask.shape}")
    print(f"  only 0/1 values: {binary}")
    print(f"  valid left-padded rows: {(left_padded & has_real_steps).sum()}/{len(mask)}")

    # Show one sequence that actually has padding.
    i = np.flatnonzero(mask[:, 0] == 0)[0]
    first_real = np.flatnonzero(mask[i] == 1)[0]
    print(f"  example row {i}: {first_real} zeros, {mask.shape[1] - first_real} ones")
    print(f"  around boundary: {mask[i, first_real-3:first_real+3].astype(int).tolist()}")

train: shape=(6427, 224)
  only 0/1 values: True
  valid left-padded rows: 6427/6427
  example row 0: 167 zeros, 57 ones
  around boundary: [0, 0, 0, 1, 1, 1]
val: shape=(1627, 224)
  only 0/1 values: True
  valid left-padded rows: 1627/1627
  example row 0: 156 zeros, 68 ones
  around boundary: [0, 0, 0, 1, 1, 1]


In [4]:
# Fold 0 timing run: setup and hyperparameters
from pathlib import Path
import copy
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

repo = Path.cwd()
while not (repo / "data/ready_full/fold0").is_dir():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from model.baseline import FullSensorCNNGRU
from model.evaluation import competition_score

FOLD = 0
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
MIN_LEARNING_RATE = 1e-5
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0  # Simple and reliable for Windows notebooks.

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; this timing run is intended for your GPU.")
device = torch.device("cuda")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Fold {FOLD}, seed {SEED}, device: {torch.cuda.get_device_name(0)}")


Fold 0, seed 42, device: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
# Load only the selected fold. Memory mapping avoids copying the ~2 GB training array into RAM.
class GestureDataset(Dataset):
    def __init__(self, fold_dir, split):
        self.x = np.load(fold_dir / f"X_{split}.npy", mmap_mode="r")
        self.mask = np.load(fold_dir / f"mask_{split}.npy", mmap_mode="r")
        self.y = np.load(fold_dir / f"y_gesture_{split}.npy", mmap_mode="r")
        if not (len(self.x) == len(self.mask) == len(self.y)):
            raise ValueError(f"Mismatched {split} array lengths")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        # Make writable per-sample copies before converting read-only memmap slices to tensors.
        x = torch.from_numpy(np.array(self.x[index], dtype=np.float32, copy=True))
        mask = torch.from_numpy(np.array(self.mask[index], copy=True))
        y = torch.tensor(int(self.y[index]), dtype=torch.long)
        return x, mask, y

fold_dir = repo / "data/ready_full" / f"fold{FOLD}"
train_dataset = GestureDataset(fold_dir, "train")
val_dataset = GestureDataset(fold_dir, "val")
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_dataset)} sequences; validation: {len(val_dataset)} sequences")


Train: 6427 sequences; validation: 1627 sequences


In [6]:
# Single 18-class baseline head; the evaluation function collapses non-target classes.
model = FullSensorCNNGRU(num_classes=18).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Trainable parameters: 524,946


In [7]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, mask, y in loader:
        x = x.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x, mask)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

@torch.inference_mode()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []
    for x, mask, y in loader:
        x = x.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x, mask)
        loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        all_true.append(y.cpu().numpy())
        all_pred.append(logits.argmax(dim=1).cpu().numpy())

    metrics = competition_score(np.concatenate(all_true), np.concatenate(all_pred))
    return total_loss / len(loader.dataset), metrics


# Training duration estimate

Run training on fold0 only, to estimate total time needed for cross validation across all folds

In [8]:
# Run ONE fold. Stop when the competition score has not improved for PATIENCE epochs.
history = []
best_score = float("-inf")
best_epoch = 0
best_weights = None
epochs_without_improvement = 0

torch.cuda.synchronize()
run_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, metrics = validate(model, val_loader, criterion)
    scheduler.step()
    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **metrics,
        "seconds": epoch_seconds,
    })
    print(
        f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
        f"| binary F1 {metrics['binary_f1']:.4f} | macro F1 {metrics['macro_f1']:.4f} "
        f"| score {metrics['score']:.4f} | {epoch_seconds:.1f}s"
    )

    if metrics["score"] > best_score:
        best_score = metrics["score"]
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

fold_seconds = time.perf_counter() - run_start
model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch}; best validation score: {best_score:.4f}")
print(f"Fold {FOLD} training + validation time: {fold_seconds / 60:.1f} minutes")


Epoch 01 | train loss 1.7105 | val loss 1.3784 | binary F1 0.9443 | macro F1 0.4516 | score 0.6979 | 35.0s
Epoch 02 | train loss 1.1426 | val loss 1.1111 | binary F1 0.9702 | macro F1 0.5660 | score 0.7681 | 22.6s
Epoch 03 | train loss 0.9311 | val loss 0.9183 | binary F1 0.9716 | macro F1 0.6069 | score 0.7892 | 15.2s
Epoch 04 | train loss 0.7863 | val loss 0.8925 | binary F1 0.9738 | macro F1 0.6115 | score 0.7927 | 16.1s
Epoch 05 | train loss 0.6948 | val loss 0.8467 | binary F1 0.9829 | macro F1 0.6320 | score 0.8075 | 16.7s
Epoch 06 | train loss 0.6183 | val loss 0.9187 | binary F1 0.9778 | macro F1 0.5964 | score 0.7871 | 16.0s
Epoch 07 | train loss 0.5452 | val loss 0.9444 | binary F1 0.9770 | macro F1 0.6144 | score 0.7957 | 15.1s
Epoch 08 | train loss 0.4870 | val loss 0.9761 | binary F1 0.9799 | macro F1 0.6205 | score 0.8002 | 13.3s
Epoch 09 | train loss 0.4373 | val loss 0.9572 | binary F1 0.9830 | macro F1 0.6263 | score 0.8046 | 13.9s
Epoch 10 | train loss 0.3776 | val lo

In [9]:
# Rough projection. Fold sizes and early-stopping epochs can differ.
print(f"One fold, one seed: {fold_seconds / 60:.1f} minutes")
print(f"Five folds, one seed: about {5 * fold_seconds / 60:.1f} minutes")
print(f"Five folds, 2 seeds: about {10 * fold_seconds / 60:.1f} minutes")
print(f"Five folds, 3 seeds: about {15 * fold_seconds / 60:.1f} minutes")


One fold, one seed: 9.2 minutes
Five folds, one seed: about 46.1 minutes
Five folds, 2 seeds: about 92.2 minutes
Five folds, 3 seeds: about 138.3 minutes
